In [0]:

from pyspark.sql import functions as F

perfis = spark.table(
    "workspace.silver.perfis_auditados"
)

# Segmentação exploratória dos viajantes
segmentos = (
    perfis
    .withColumn(
        "orcamento_perfil",
        F.when(
            F.col("orcamento_todas_faixas"),
            "Todas as faixas"
        )
        .when(
            F.size("form_b_pt") == 1,
            F.element_at("form_b_pt", 1)
        )
        .otherwise("Não informado / outro")
    )
    .withColumn(
        "estacao_perfil",
        F.when(
            F.size("form_c_pt") == 1,
            F.element_at("form_c_pt", 1)
        )
        .otherwise("Não informada")
    )
    .withColumn(
        "experiencia_perfil",
        F.when(
            F.size("form_f_pt") == 0,
            "Não informada"
        )
        .otherwise(
            F.concat_ws(", ", F.col("form_f_pt"))
        )
    )
    .groupBy(
        "orcamento_perfil",
        "estacao_perfil",
        "experiencia_perfil"
    )
    .agg(
        F.countDistinct("id").alias("viajantes")
    )
    .orderBy(F.desc("viajantes"))
)

display(segmentos.limit(20))


In [0]:

from pyspark.sql import functions as F

perfis = spark.table("workspace.silver.perfis_traduzidos")

perfis_auditados = (
    perfis
    .withColumn(
        "regiao_inconsistente",
        (
            F.array_contains(
                F.col("form_r_codigos"), "0"
            ) &
            (F.coalesce(
                F.size("form_rr_codigos"), F.lit(0)
            ) > 0)
        ) |
        (
            F.array_contains(
                F.col("form_r_codigos"), "1"
            ) &
            (F.coalesce(
                F.size("form_rr_codigos"), F.lit(0)
            ) == 0)
        )
    )
)

display(
    perfis_auditados.groupBy(
        "regiao_inconsistente"
    ).count()
)


In [0]:

from pyspark.sql import functions as F

perfis = spark.table(
    "workspace.silver.perfis_traduzidos"
)

perfis_auditados = (
    perfis
    .withColumn(
        "orcamento_todas_faixas",
        F.size("form_b_codigos") == 4
    )
    .withColumn(
        "popularidade_todas_opcoes",
        F.size("form_j_codigos") == 3
    )
    .withColumn(
        "regiao_inconsistente",
        (
            F.array_contains(
                F.col("form_r_codigos"), "0"
            ) &
            (F.size("form_rr_codigos") > 0)
        ) |
        (
            F.array_contains(
                F.col("form_r_codigos"), "1"
            ) &
            (F.size("form_rr_codigos") == 0)
        )
    )
)

display(
    perfis_auditados.agg(
        F.sum(
            F.col("orcamento_todas_faixas").cast("int")
        ).alias("orcamento_todas_faixas"),
        F.sum(
            F.col("popularidade_todas_opcoes").cast("int")
        ).alias("popularidade_todas_opcoes"),
        F.sum(
            F.col("regiao_inconsistente").cast("int")
        ).alias("regioes_inconsistentes")
    )
)


In [0]:

from pyspark.sql import functions as F

perfis = spark.table("workspace.silver.perfis_traduzidos")

# 1. Combinações de orçamento
print("ORÇAMENTO - Respostas múltiplas")

display(
    perfis
    .filter(F.size("form_b_codigos") > 1)
    .groupBy("form_b")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
)

# 2. Combinações de popularidade
print("POPULARIDADE - Respostas múltiplas")

display(
    perfis
    .filter(F.size("form_j_codigos") > 1)
    .groupBy("form_j")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
)

# 3. Relação entre as perguntas geográficas
print("REGIÕES - Consistência entre FORM_R e FORM_RR")

display(
    perfis
    .withColumn(
        "resposta_r",
        F.when(F.size("form_r_codigos") == 0, "Sem resposta")
         .otherwise(
             F.concat_ws(", ", F.col("form_r_pt"))
         )
    )
    .withColumn(
        "respondeu_rr",
        F.when(F.size("form_rr_codigos") > 0, "Sim")
         .otherwise("Não")
    )
    .groupBy("resposta_r", "respondeu_rr")
    .count()
    .orderBy("resposta_r", "respondeu_rr")
)


In [0]:

from pyspark.sql import functions as F

perfis = spark.table("workspace.silver.perfis_traduzidos")

campos = [
    "form_a", "form_b", "form_c", "form_f",
    "form_g", "form_h", "form_i", "form_j",
    "form_r", "form_rr"
]

auditoria = []

for campo in campos:
    codigos = F.col(campo + "_codigos")

    linha = perfis.agg(
        F.count("*").alias("total"),
        F.sum(
            F.when(F.size(codigos) > 0, 1).otherwise(0)
        ).alias("respondidos"),
        F.sum(
            F.when(F.size(codigos) > 1, 1).otherwise(0)
        ).alias("multiplas_respostas")
    ).first()

    auditoria.append((
        campo,
        linha["total"],
        linha["respondidos"],
        linha["total"] - linha["respondidos"],
        linha["multiplas_respostas"]
    ))

display(
    spark.createDataFrame(
        auditoria,
        [
            "campo",
            "total_viajantes",
            "respondidos",
            "sem_resposta",
            "multiplas_respostas"
        ]
    )
)


In [0]:

from pyspark.sql import functions as F

perfis = spark.table("workspace.silver.perfis_traduzidos")

campos = {
    "form_a_pt": "Faixa etária do grupo",
    "form_b_pt": "Orçamento",
    "form_c_pt": "Estação do ano",
    "form_f_pt": "Experiências desejadas",
    "form_g_pt": "Paisagens",
    "form_h_pt": "Nível de atividade",
    "form_i_pt": "Segurança",
    "form_j_pt": "Popularidade do destino",
    "form_r_pt": "Abrangência geográfica",
    "form_rr_pt": "Regiões desejadas"
}

total_viajantes = perfis.select("id").distinct().count()

resultados = []

for coluna, pergunta in campos.items():
    distribuicao = (
        perfis
        .select(
            "id",
            F.explode_outer(F.col(coluna)).alias("resposta")
        )
        .filter(F.col("resposta").isNotNull())
        .groupBy("resposta")
        .agg(F.countDistinct("id").alias("viajantes"))
        .withColumn("pergunta", F.lit(pergunta))
        .withColumn(
            "percentual",
            F.round(
                F.col("viajantes") / F.lit(total_viajantes) * 100,
                2
            )
        )
        .select("pergunta", "resposta", "viajantes", "percentual")
    )
    resultados.append(distribuicao)

eda_perfil = resultados[0]

for resultado in resultados[1:]:
    eda_perfil = eda_perfil.unionByName(resultado)

display(
    eda_perfil.orderBy("pergunta", F.desc("viajantes"))
)
